# 03 – แผนที่พื้นที่เหมาะกับพืช (โมเดลล่าสุด + ข้อมูลดินทั่วประเทศ)
ใช้โมเดลล่าสุดจาก repo ของเพื่อน (`models/enriched_crop_pipeline.pkl` — Random Forest, 23 features)
ทำนายพืชที่เหมาะกับ **ทุกจุดตรวจดินจริง** ของกรมพัฒนาที่ดิน (~126,000 จุด ครบ 77 จังหวัด) แล้วสรุปเป็นรายอำเภอ

| ส่วน | เนื้อหา |
|---|---|
| 1 | โหลดโมเดลและข้อมูล Kaggle |
| 2 | ประเมินโมเดลบนชุดทดสอบ (Accuracy, Precision, Recall, F1, Top-3) |
| 3 | อากาศจากสถานีกรมอุตุฯ |
| 4 | ดินทั่วประเทศ (osd01 ผ่าน SISoil) |
| 5 | ทำนายทุกจุดตรวจดิน |
| 6 | สรุปรายอำเภอ |
| 7 | ตรวจกับพื้นที่ปลูกจริง (F1 ระดับจังหวัด) |
| 8 | แผนที่รายอำเภอแบบมีเมนู |

รันจากโฟลเดอร์ `notebooks/` · ผลลัพธ์ทั้งหมดถูกบันทึกในโฟลเดอร์ `outputs/`

In [ ]:
# ติดตั้งแพ็กเกจ (รันครั้งแรกครั้งเดียว)
import sys
PKGS = ['folium', 'pandas', 'scikit-learn', 'joblib', 'requests', 'kagglehub', 'xgboost']
if 'pyodide' in sys.modules:
    import piplite
    for p in PKGS:
        try:
            await piplite.install(p)
        except Exception:
            print('ติดตั้งไม่ได้ (ข้ามได้):', p)
else:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS])
print('ติดตั้งเสร็จ')

## 1. ตั้งค่าโฟลเดอร์ โหลดโมเดลและข้อมูล Kaggle

In [ ]:
import json, glob, os, warnings
from pathlib import Path
import numpy as np, pandas as pd, joblib
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')

# โฟลเดอร์ของโปรเจค (รันจาก notebooks/ หรือจากโฟลเดอร์หลักก็ได้)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW, REF, SOIL_DIR = ROOT / 'data' / 'raw', ROOT / 'data' / 'reference', ROOT / 'data' / 'soil'
MODELS, OUT = ROOT / 'models', ROOT / 'outputs'
SOIL_DIR.mkdir(parents=True, exist_ok=True); OUT.mkdir(exist_ok=True)

try:
    import kagglehub
    path = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")
    kag = pd.read_csv(glob.glob(path + '/*.csv')[0]); print("โหลด Kaggle จาก:", path)
except Exception:
    kag = pd.read_csv(RAW / 'Crop_recommendation.csv'); print("ใช้ data/raw/Crop_recommendation.csv แทน")

PIPELINE = MODELS / 'enriched_crop_pipeline.pkl'
pipe = joblib.load(PIPELINE)
model, scaler, le, MODEL_FEATS = pipe['model'], pipe['scaler'], pipe['label_encoder'], list(pipe['feature_names'])
FEATS = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']     # 7 ปัจจัยหลัก
LOC_FEATS = [f for f in MODEL_FEATS if f.startswith('location_')]
CROPS = list(le.classes_)
print(type(model).__name__, '|', len(MODEL_FEATS), 'features |', len(LOC_FEATS), 'คอลัมน์ location |', len(CROPS), 'พืช')

## 2. ประเมินโมเดลบนชุดทดสอบ
แบ่งข้อมูลแบบเดียวกับ notebook 02 ของเพื่อน (80:20, stratified, `random_state=42`) แล้ววัดผลโมเดลที่โหลดมา
Top-3 hit rate = สัดส่วนที่พืชที่ถูกต้องอยู่ใน 3 อันดับแรกที่โมเดลแนะนำ (ใช้เป็น Precision@3 / Recall@3 ของงานแนะนำ)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

TRAIN_CSV = REF / 'enriched_crop_dataset_cleaned.csv'
if TRAIN_CSV.exists():
    df = pd.read_csv(TRAIN_CSV)
    y = le.transform(df['label'])
    X = pd.get_dummies(df.drop(columns=['label']), drop_first=True).reindex(columns=MODEL_FEATS, fill_value=0)
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    Xs = scaler.transform(X_test)
    y_pred = model.predict(Xs); top3 = np.argsort(-model.predict_proba(Xs), axis=1)[:, :3]
    test_metrics = pd.DataFrame([{
        'Model': type(model).__name__, 'Test rows': len(y_test),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (macro)': precision_score(y_test, y_pred, average='macro'),
        'Recall (macro)': recall_score(y_test, y_pred, average='macro'),
        'F1 (macro)': f1_score(y_test, y_pred, average='macro'),
        'Top-3 hit rate': (top3 == y_test[:, None]).any(axis=1).mean()}]).round(4)
    test_metrics.to_csv(OUT / 'model_test_metrics.csv', index=False, encoding='utf-8-sig')
    display(test_metrics) if 'display' in globals() else print(test_metrics.to_string(index=False))
else:
    print('ไม่พบ', TRAIN_CSV, '— ข้ามการประเมินบนชุดทดสอบ')

## 3. อากาศจากสถานีกรมอุตุฯ (ค่าปกติ 1981–2010)

In [ ]:
rows = []
for s in ET.parse(RAW / 'climate_normal.xml').getroot().iter('StationClimateNormal'):
    nv = s.find('NormalValue'); el = nv.find('Element')
    rows.append(dict(station=s.findtext('StationName'),
                     lat=pd.to_numeric(s.findtext('Latitude'), errors='coerce'),
                     lon=pd.to_numeric(s.findtext('Longitude'), errors='coerce'),
                     element=el.text, vals=[pd.to_numeric(c.text, errors='coerce') for c in nv if c.tag.startswith('ValueOn')]))
clim = pd.DataFrame(rows).dropna(subset=['lat'])

def wide(name):   # ตาราง สถานี x 12 เดือน
    d = clim[clim.element == name].drop_duplicates('station').set_index('station')
    return pd.DataFrame(d.vals.tolist(), index=d.index)
T, H, R = wide('Mean Dry-bulb Temperature'), wide('Mean Relative Humidity'), wide('Total Rainfall')
st = clim.drop_duplicates('station').set_index('station')[['lat', 'lon']]
st = st.loc[st.index.intersection(T.dropna().index).intersection(H.dropna().index).intersection(R.dropna().index)]
print('สถานีที่ข้อมูลครบ:', len(st))

## 4. ดินทั่วประเทศ (osd01 ผ่าน SISoil)
ครั้งแรกดาวน์โหลด `sites.csv` จาก [SISoil](https://github.com/Navavit/sisoil) (~35 MB) แล้วเก็บฉบับคัดแล้วที่ `data/soil/thailand_soil_samples.csv`
ตั้ง `REFRESH = True` เพื่อดึงฉบับล่าสุด · ไม่มีค่า N ที่วัดจริง จึงประมาณ `N = OM% × 20` (แก้ได้ที่ `N_PER_OM`)

In [ ]:
SISOIL_URL = 'https://raw.githubusercontent.com/Navavit/sisoil/main/data/osd/sites.csv'
SOIL_FILE = SOIL_DIR / 'thailand_soil_samples.csv'
REFRESH = False
N_PER_OM = 20

if SOIL_FILE.exists() and not REFRESH:
    soil = pd.read_csv(SOIL_FILE, encoding='utf-8-sig', low_memory=False); print('ใช้', SOIL_FILE.name, 'ที่เก็บไว้')
else:
    print('กำลังดาวน์โหลดจาก SISoil ...')
    raw = pd.read_csv(SISOIL_URL, encoding='utf-8-sig', low_memory=False)
    bad = {'no_coords', 'off_land', 'far_from_province', 'value_out_of_range', 'date_future'}
    flags = raw.qa_flags.fillna('').str.split(';')
    ok = (raw[['ph', 'om_pct', 'p_mg_kg', 'k_mg_kg']].notna().all(axis=1)
          & ~flags.apply(lambda x: bool(bad & set(x)))
          & raw.lat.between(5, 21) & raw.lon.between(97, 106))
    soil = (raw[ok].rename(columns={'amphur': 'amphoe', 'district': 'tambon', 'ph': 'pH', 'om_pct': 'OM_pct',
                                    'p_mg_kg': 'P_mg_kg', 'k_mg_kg': 'K_mg_kg', 'ec_ds_m': 'EC_dS_m'})
            [['id', 'date', 'province', 'amphoe', 'tambon', 'lat', 'lon', 'n_samples', 'pH', 'EC_dS_m', 'OM_pct', 'P_mg_kg', 'K_mg_kg', 'qa_flags']])
    soil.to_csv(SOIL_FILE, index=False, encoding='utf-8-sig')
    print(f'เก็บ {len(soil):,} จาก {len(raw):,} จุด')
soil = soil.reset_index(drop=True)
soil['N'] = soil.OM_pct * N_PER_OM
print(f"จุดตรวจดิน: {len(soil):,} | จังหวัด: {soil.province.nunique()} | อำเภอ: {soil.groupby(['province','amphoe']).ngroups}")

## 5. ทำนายทุกจุดตรวจดิน
- อากาศ: สถานีที่ใกล้จุดตรวจที่สุด · ค่าที่เกินช่วง Kaggle ถูกตัดให้อยู่ในช่วงก่อนเข้าโมเดล
- `location_*` (มีในโมเดลล่าสุด): ใช้จังหวัดของจุดตรวจ ถ้าโมเดลไม่รู้จักจังหวัดนั้น ใช้จังหวัดที่โมเดลรู้จักซึ่งอยู่ใกล้ที่สุด
- โค้ดสร้างคอลัมน์ตาม `feature_names` ของโมเดล จึงใช้กับโมเดลรุ่นอื่นของเพื่อนได้ด้วย ถ้ายังใช้ชุดคอลัมน์แบบเดียวกัน

In [ ]:
def hav(la1, lo1, la2, lo2):
    la1, lo1, la2, lo2 = map(np.radians, (la1, lo1, la2, lo2))
    a = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

def nearest(lat, lon, rlat, rlon, chunk=20000):
    idx, dist = np.empty(len(lat), int), np.empty(len(lat))
    for i in range(0, len(lat), chunk):
        D = hav(lat[i:i+chunk, None], lon[i:i+chunk, None], np.asarray(rlat)[None], np.asarray(rlon)[None])
        idx[i:i+chunk], dist[i:i+chunk] = D.argmin(1), D.min(1)
    return idx, dist

i_st, d_st = nearest(soil.lat.values, soil.lon.values, st.lat.values, st.lon.values)
soil['station'] = st.index[i_st]; soil['station_km'] = d_st.round(1)
print(f'ระยะจุดตรวจดิน–สถานีอากาศ: มัธยฐาน {np.median(d_st):.0f} กม., 90% ไม่เกิน {np.percentile(d_st, 90):.0f} กม.')

# จังหวัด (ภาษาอังกฤษตามที่โมเดลใช้) ของแต่ละจุดตรวจ
THAI_TO_ENG = {"กำแพงเพชร": "KAMPHAENG PHET", "ตรัง": "TRANG", "ตาก": "TAK", "นครราชสีมา": "NAKHON RATCHASIMA",
               "นครสวรรค์": "NAKHON SAWAN", "ปทุมธานี": "PATHUM THANI", "พะเยา": "PHAYAO", "ราชบุรี": "RATCHABURI",
               "ร้อยเอ็ด": "ROI ET", "สงขลา": "SONGKHLA", "สุโขทัย": "SUKHOTHAI", "อุดรธานี": "UDON THANI",
               "อุทัยธานี": "UTHAI THANI", "อุบลราชธานี": "UBON RATCHATHANI", "เชียงราย": "CHIANG RAI",
               "เชียงใหม่": "CHIANG MAI", "เพชรบุรี": "PHETCHABURI", "แพร่": "PHRAE", "แม่ฮ่องสอน": "MAE HONG SON"}
PROV_GEO = json.load(open(REF / 'thailand_provinces.geojson', encoding='utf-8')) if (REF / 'thailand_provinces.geojson').exists() else None
soil['location'] = ''
if LOC_FEATS:
    KNOWN = sorted(pd.read_csv(TRAIN_CSV).location.unique()) if TRAIN_CSV.exists() else [f[9:] for f in LOC_FEATS]
    cent = {}
    for f in PROV_GEO['features']:                         # จุดกึ่งกลางคร่าว ๆ ของแต่ละจังหวัด
        g = f['geometry']; ring = (g['coordinates'][0][0] if g['type'] == 'MultiPolygon' else g['coordinates'][0])
        cent[f['properties']['pro_th']] = (np.mean([p[1] for p in ring]), np.mean([p[0] for p in ring]))
    known_th = [th for th, en in THAI_TO_ENG.items() if en in KNOWN and th in cent]
    loc_of = {}
    for th in soil.province.unique():
        if THAI_TO_ENG.get(th) in KNOWN: loc_of[th] = THAI_TO_ENG[th]; continue
        la, lo = cent.get(th, (soil[soil.province == th].lat.median(), soil[soil.province == th].lon.median()))
        j = int(np.argmin([hav(la, lo, *cent[k]) for k in known_th])); loc_of[th] = THAI_TO_ENG[known_th[j]]
    soil['location'] = soil.province.map(loc_of)
    print(f"จังหวัดที่โมเดลรู้จัก: {len(KNOWN)} · จังหวัดอื่นใช้จังหวัดที่รู้จักซึ่งใกล้ที่สุด")

SEASONS = {'rainy': [4,5,6,7,8,9], 'dry': [10,11,0,1,2,3], 'year': list(range(12))}
SEASON_TH = {'rainy': 'ฤดูฝน (พ.ค.–ต.ค.)', 'dry': 'ฤดูแล้ง (พ.ย.–เม.ย.)', 'year': 'ทั้งปี'}

def features(season):
    m = SEASONS[season]; s_ = soil.station
    return pd.DataFrame({'N': soil.N, 'P': soil.P_mg_kg, 'K': soil.K_mg_kg,
                         'temperature': T.loc[s_, m].mean(axis=1).values, 'humidity': H.loc[s_, m].mean(axis=1).values,
                         'ph': soil.pH, 'rainfall': R.loc[s_, m].mean(axis=1).values})[FEATS]

def predict(season):
    X = features(season); Xc = X.clip(kag[FEATS].min(), kag[FEATS].max(), axis=1)
    annual = {'temperature_xml': T.loc[soil.station].mean(axis=1).values, 'humidity_xml': H.loc[soil.station].mean(axis=1).values,
              'rainfall_xml': R.loc[soil.station].sum(axis=1).values}
    M = pd.DataFrame(index=soil.index, columns=MODEL_FEATS, dtype=float)
    for f in MODEL_FEATS:
        if f in Xc: M[f] = Xc[f].values
        elif f.endswith('_original'): M[f] = Xc[f[:-len('_original')]].values
        elif f in annual: M[f] = annual[f]
        elif f.startswith('location_'): M[f] = (soil.location.values == f[len('location_'):]).astype(float)
        else: raise ValueError(f'ไม่รู้วิธีสร้าง feature: {f}')
    proba = model.predict_proba(scaler.transform(M))
    top = np.argsort(-proba, axis=1)[:, :3]
    return X, top, proba[np.arange(len(soil))[:, None], top]

PRED = {}
for s_ in SEASONS:
    X, top, pr = predict(s_)
    PRED[s_] = dict(X=X, top=top, prob=pr)
    print(SEASON_TH[s_], '— พืชอันดับ 1 ที่พบบ่อย:', pd.Series(np.array(CROPS)[top[:, 0]]).value_counts().head(5).to_dict())

## 6. สรุปรายอำเภอ
"เหมาะ" = พืชนั้นติด 3 อันดับแรกของโมเดลที่จุดตรวจนั้น · บันทึก `outputs/soil_predictions.csv` (รายจุด) และ `outputs/amphoe_crop_suitability.csv` (รายอำเภอ)

In [ ]:
KEY = ['province', 'amphoe']
def share_table(season, level=KEY):
    top = PRED[season]['top']
    hit = np.zeros((len(soil), len(CROPS)), dtype=np.float32)
    for k in range(3): hit[np.arange(len(soil)), top[:, k]] = 1
    return pd.DataFrame(hit, columns=CROPS).groupby([soil[c] for c in level]).mean()

SHARE = {s_: share_table(s_) for s_ in SEASONS}
amp = soil.groupby(KEY).agg(samples=('id', 'size'), lat=('lat', 'median'), lon=('lon', 'median'),
                            N=('N', 'median'), P=('P_mg_kg', 'median'), K=('K_mg_kg', 'median'), ph=('pH', 'median'))

out = soil[['id', 'date', 'province', 'amphoe', 'tambon', 'lat', 'lon', 'pH', 'OM_pct', 'P_mg_kg', 'K_mg_kg', 'N', 'station', 'station_km']].copy()
for s_ in SEASONS:
    for k in range(3): out[f'{s_}_top{k+1}'] = np.array(CROPS)[PRED[s_]['top'][:, k]]
out.to_csv(OUT / 'soil_predictions.csv', index=False, encoding='utf-8-sig')
amp.join((SHARE['rainy'] * 100).round(1).add_prefix('rainy_pct_')).to_csv(OUT / 'amphoe_crop_suitability.csv', encoding='utf-8-sig')
print('บันทึก outputs/soil_predictions.csv และ outputs/amphoe_crop_suitability.csv')

CROP = 'rice'      # เปลี่ยนพืชเพื่อดูอำเภอที่เหมาะที่สุด
print(f'\n10 อำเภอที่จุดตรวจดินเหมาะกับ {CROP} มากที่สุด (ฤดูฝน, อำเภอที่มีจุดตรวจ ≥ 30 จุด):')
t = amp.join(SHARE['rainy'][CROP].rename('pct') * 100)
t[t.samples >= 30].sort_values('pct', ascending=False).head(10)[['samples', 'pct', 'N', 'P', 'K', 'ph']].round(1)

## 7. ตรวจกับพื้นที่ปลูกจริง (F1 ระดับจังหวัด)
จังหวัด "ปลูกจริง" เมื่อเนื้อที่ถึงเกณฑ์ใน `ACTUAL` (สถิติ สศก.) · โมเดล "แนะนำ" เมื่อจุดตรวจดินในจังหวัด ≥ `MIN_SHARE` ให้พืชนั้นติด 3 อันดับแรก
F1 = ค่าเฉลี่ยฮาร์มอนิกของ "แนะนำถูก" (precision) และ "จับได้" (recall)

In [ ]:
MIN_SHARE = 0.25
ACTUAL = {'rice (นาปี)':   ('rice',    'rice_rai_2567',     100000, 'rainy'),
          'rice (นาปรัง)': ('rice',    'dry_rice_rai_2568', 100000, 'dry'),
          'maize':         ('maize',   'maize_rai_2567',     10000, 'rainy'),
          'coffee':        ('coffee',  'coffee_rai_2568',     1000, 'rainy'),
          'coconut':       ('coconut', 'coconut_rai_2568',    1000, 'rainy')}
truth = pd.read_csv(REF / 'thailand_actual_crop_area.csv', index_col=0) if (REF / 'thailand_actual_crop_area.csv').exists() else None
if truth is None:
    print('ไม่พบ data/reference/thailand_actual_crop_area.csv — ข้ามการตรวจ')
else:
    rows = []
    for label, (crop, col, thr, season) in ACTUAL.items():
        if col not in truth: continue
        share = share_table(season, ['province'])[crop]
        hit = share >= MIN_SHARE
        act = truth[col].reindex(share.index).fillna(0) >= thr
        tp = int((hit & act).sum()); p_ = tp / hit.sum() if hit.sum() else 0.0; r_ = tp / act.sum() if act.sum() else 0.0
        rows.append({'พืช': label, 'ฤดูที่เทียบ': season, 'ปลูกจริง': int(act.sum()), 'โมเดลแนะนำ': int(hit.sum()), 'ตรงกัน': tp,
                     'Precision': round(p_, 2), 'Recall': round(r_, 2), 'F1': round(2 * p_ * r_ / (p_ + r_), 2) if p_ + r_ else 0.0})
    eval_df = pd.DataFrame(rows); eval_df.to_csv(OUT / 'model_check_actual_area.csv', index=False, encoding='utf-8-sig')
    display(eval_df) if 'display' in globals() else print(eval_df.to_string(index=False))
    print(f'F1 เฉลี่ย: {eval_df.F1.mean():.2f}')

## 8. แผนที่รายอำเภอ (มีเมนูเลือกพืช ฤดู และเกณฑ์)
หมุด = อำเภอ · สีขอบ: เขียวเข้ม ≥ 50% ของจุดตรวจเหมาะ, เขียว 25–50%, ส้ม ต่ำกว่า 25% · กดหมุดดูค่าดินมัธยฐานเทียบช่วงที่พืชต้องการ
**แนะนำให้ดูช่วง "ฤดูฝน"** เพราะ "ทั้งปี" เฉลี่ยฝนรวมเดือนแล้ง ทำให้พืชที่ต้องการน้ำมาก เช่น ข้าว หายจากภาคกลางและอีสาน
บันทึกเป็น `outputs/crop_suitability_map.html` เปิดในเบราว์เซอร์ได้โดยไม่ต้องมี Jupyter

In [ ]:
import folium
from branca.element import Element
ICON = {'apple':'🍎','banana':'🍌','blackgram':'🫘','chickpea':'🫘','coconut':'🥥','coffee':'☕','cotton':'☁️','grapes':'🍇',
        'jute':'🌿','kidneybeans':'🫘','lentil':'🫘','maize':'🌽','mango':'🥭','mothbeans':'🫘','mungbean':'🫛','muskmelon':'🍈',
        'orange':'🍊','papaya':'🧡','pigeonpeas':'🫛','pomegranate':'🔴','rice':'🌾','watermelon':'🍉'}
TH = {'apple':'แอปเปิล','banana':'กล้วย','blackgram':'ถั่วดำ','chickpea':'ถั่วลูกไก่','coconut':'มะพร้าว','coffee':'กาแฟ','cotton':'ฝ้าย',
      'grapes':'องุ่น','jute':'ปอกระเจา','kidneybeans':'ถั่วแดง','lentil':'ถั่วเลนทิล','maize':'ข้าวโพด','mango':'มะม่วง','mothbeans':'ถั่วมอธ',
      'mungbean':'ถั่วเขียว','muskmelon':'แคนตาลูป','orange':'ส้ม','papaya':'มะละกอ','pigeonpeas':'ถั่วแระ','pomegranate':'ทับทิม',
      'rice':'ข้าว','watermelon':'แตงโม'}

MAP_JS = """
window.addEventListener('load', function () {
  var map = __MAP__, D = __DATA__, layer = L.layerGroup().addTo(map);
  var FN = ['ไนโตรเจน N (ประมาณ)', 'ฟอสฟอรัส P', 'โพแทสเซียม K', 'อุณหภูมิ °C', 'ความชื้น %', 'pH', 'ฝน มม./เดือน'];
  var ctl = L.control({position: 'topright'});
  ctl.onAdd = function () {
    var d = L.DomUtil.create('div');
    d.style.cssText = 'background:#fff;color:#222;padding:8px 10px;border-radius:6px;box-shadow:0 1px 5px rgba(0,0,0,.4);font:14px sans-serif;line-height:1.9';
    d.innerHTML = 'พืช <select id="cropSel"></select><br>ช่วง <select id="seaSel"></select><br>' +
      'แสดงอำเภอที่เหมาะ ≥ <select id="minSel"><option value="10">10%</option><option value="25" selected>25%</option><option value="50">50%</option></select> ของจุดตรวจ' +
      '<br><label id="actLbl"><input type="checkbox" id="actChk"> เฉพาะจังหวัดที่ปลูกจริง</label><div id="cnt" style="color:#555;font-size:12px"></div>';
    L.DomEvent.disableClickPropagation(d); L.DomEvent.disableScrollPropagation(d); return d;
  };
  ctl.addTo(map);
  var cs = document.getElementById('cropSel'), ss = document.getElementById('seaSel'), ms = document.getElementById('minSel');
  var chk = document.getElementById('actChk'), lbl = document.getElementById('actLbl'), cnt = document.getElementById('cnt');
  Object.keys(D.sea).forEach(function (k) { ss.add(new Option(D.sea[k], k)); });
  function topOf(p, s) { var b = null, v = -1; for (var c in p.s[s]) if (p.s[s][c] > v) { v = p.s[s][c]; b = c; } return [b, v]; }
  function grown(p, crop, s) { var a = D.act[crop]; if (!a) return null; var k = (crop === 'rice' && s === 'dry' && a.dry) ? 'dry' : 'main'; return a[k][p.pv] === 1; }
  function fill() {
    var cur = cs.value || '__CROP__', min = +ms.value;
    var n = {}; D.crops.forEach(function (c) { n[c] = D.pts.filter(function (p) { return (p.s[ss.value][c] || 0) >= min; }).length; });
    cs.innerHTML = ''; cs.add(new Option('ทุกพืช (พืชที่เหมาะที่สุดของแต่ละอำเภอ)', 'all'));
    D.crops.slice().sort(function (a, b) { return n[b] - n[a]; }).forEach(function (c) { cs.add(new Option(D.icon[c] + ' ' + D.th[c] + ' (' + n[c] + ' อำเภอ)', c)); });
    cs.value = cur;
  }
  function draw() {
    layer.clearLayers(); var shown = 0, crop = cs.value, s = ss.value, min = +ms.value;
    D.pts.forEach(function (p) {
      var c, pct;
      if (crop === 'all') { var t = topOf(p, s); c = t[0]; pct = t[1]; if (!c) return; }
      else { c = crop; pct = p.s[s][c] || 0; if (pct < min) return; }
      var g = grown(p, c, s); if (chk.checked && g === false) return;
      shown++;
      var col = pct >= 50 ? '#1f6e43' : pct >= 25 ? '#3fa36b' : '#c98a2b';
      var html = '<div style="width:24px;height:24px;border-radius:50%;background:#fff;border:3px solid ' + col + ';text-align:center;line-height:18px;font-size:13px">' + D.icon[c] + '</div>';
      var R = D.rng[c], ok = 0, rows = '';
      p.v[s].forEach(function (v, i) {
        var lo = R[i][0], hi = R[i][1], st = v < lo ? 'ต่ำไป' : v > hi ? 'สูงไป' : 'เข้าช่วง'; if (st === 'เข้าช่วง') ok++;
        rows += '<tr><td>' + FN[i] + '</td><td style="text-align:right;padding:0 8px">' + v + '</td><td style="padding-right:8px">' + lo + '–' + hi +
                '</td><td style="font-weight:bold;color:' + (st === 'เข้าช่วง' ? '#1f6e43' : '#b7791f') + '">' + st + '</td></tr>';
      });
      var best = Object.keys(p.s[s]).sort(function (a, b) { return p.s[s][b] - p.s[s][a]; }).slice(0, 3)
                 .map(function (k, i) { return (i + 1) + '. ' + D.th[k] + ' ' + p.s[s][k] + '%'; }).join('<br>');
      var pop = '<b>อ.' + p.a + ' จ.' + p.pv + '</b> · จุดตรวจดิน ' + p.n + ' จุด<br><b>' + D.th[c] + ': เหมาะ ' + pct + '% ของจุดตรวจ</b>' +
        '<hr style="margin:4px 0">พืชที่เหมาะที่สุดในอำเภอนี้<br>' + best +
        '<hr style="margin:4px 0">ค่ามัธยฐานของอำเภอเทียบช่วงของ' + D.th[c] + ' (เข้าช่วง ' + ok + '/7)' +
        '<table style="font-size:12px;border-collapse:collapse;white-space:nowrap">' + rows + '</table>' +
        '<span style="color:#666">ช่วงที่พืชต้องการ = ต่ำสุด–สูงสุดของพืชนั้นในชุดข้อมูล Kaggle</span>' +
        (g === null ? '' : '<br><b>จ.' + p.pv + (g ? ' ปลูก' : ' ไม่ได้ปลูก') + D.th[c] + 'จริงตามสถิติ สศก.</b>');
      L.marker([p.la, p.lo], {icon: L.divIcon({html: html, className: '', iconSize: [24, 24], iconAnchor: [12, 12]}), zIndexOffset: pct})
        .bindTooltip('อ.' + p.a + ' – ' + D.th[c] + ' ' + pct + '%').bindPopup(pop, {maxWidth: 400}).addTo(layer);
    });
    var has = crop !== 'all' && !!D.act[crop]; chk.disabled = !has; lbl.style.color = has ? '#222' : '#aaa';
    cnt.textContent = shown + ' อำเภอ · ขอบเขียวเข้ม ≥50%, เขียว 25–50%, ส้ม <25%' + (has ? '' : ' · พืชนี้ยังไม่มีสถิติปลูกจริง');
  }
  ss.value = '__SEASON__'; fill(); draw();
  cs.onchange = draw; chk.onchange = draw; ss.onchange = function () { fill(); draw(); }; ms.onchange = function () { fill(); draw(); };
});
"""

def build_map(crop='rice', season='rainy'):
    m = folium.Map(location=[13.2, 101.2], zoom_start=6, tiles=None)
    folium.TileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
                     attr='Tiles &copy; Esri', name='แผนที่ฐาน (Esri)').add_to(m)
    if PROV_GEO:
        folium.GeoJson(PROV_GEO, name='ขอบเขตจังหวัด', tooltip=folium.GeoJsonTooltip(['pro_th'], labels=False),
                       style_function=lambda f: {'color': '#7d8f80', 'weight': 1, 'fillColor': '#e3eadf', 'fillOpacity': 0.5}).add_to(m)
    folium.LayerControl(position='bottomleft', collapsed=True).add_to(m)
    clim_med = {s_: PRED[s_]['X'].groupby([soil[c] for c in KEY])[['temperature', 'humidity', 'rainfall']].median() for s_ in SEASONS}
    pts = []
    for key, r in amp.iterrows():
        d = dict(a=key[1], pv=key[0], la=round(r.lat, 4), lo=round(r.lon, 4), n=int(r.samples), s={}, v={})
        for s_ in SEASONS:
            sh = SHARE[s_].loc[key]
            d['s'][s_] = {c: int(round(v * 100)) for c, v in sh.items() if v >= 0.01}
            cm = clim_med[s_].loc[key]
            d['v'][s_] = [round(float(x), 1) for x in (r.N, r.P, r.K, cm.temperature, cm.humidity, r.ph, cm.rainfall)]
        pts.append(d)
    act = {}
    if truth is not None:
        flag = lambda col, thr: {p: int(v >= thr) for p, v in truth[col].items()}
        for label, (c, col, thr, s_) in ACTUAL.items():
            if col in truth: act.setdefault(c, {})['dry' if (c == 'rice' and s_ == 'dry') else 'main'] = flag(col, thr)
    rng = {c: [[round(float(g[f].min()), 1), round(float(g[f].max()), 1)] for f in FEATS] for c, g in kag.groupby('label')}
    data = json.dumps(dict(pts=pts, icon=ICON, th=TH, crops=CROPS, act=act, rng=rng, sea=SEASON_TH), ensure_ascii=False)
    js = MAP_JS.replace('__MAP__', m.get_name()).replace('__DATA__', data).replace('__CROP__', crop).replace('__SEASON__', season)
    m.get_root().script.add_child(Element(js))
    return m

m = build_map('rice', 'rainy')
m.save(OUT / 'crop_suitability_map.html')
m